In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Set the size of images we want to use and how many images we process at a time
IMG_SIZE = (128, 128)
BATCH_SIZE = 16 #processes 16 images at a time (helps with memory efficiency)

data_dir = "/content/drive/My Drive/Image_identification/Train"

# Data Augmentation - This helps the model learn better by slightly changing the images
# This prevents the model from memorizing specific images and instead learns general patterns
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalizes pixel values (0-255) to a range between 0 and 1
    rotation_range=30,  # Rotates images up to 30 degrees to make sure the model can handle different angles
    width_shift_range=0.2,  # Randomly shifts images left or right by 20% to make it robust to position changes
    height_shift_range=0.2,  # Randomly shifts images up or down by 20% for the same reason
    brightness_range=[0.5, 1.5],  # Randomly makes images darker or brighter to handle lighting differences
    horizontal_flip=True,  # Flips images left to right to help with different orientations
    vertical_flip=True  # Flips images upside down to make sure the model learns from all angles
)

# This function loads the images from the folder and prepares them for training
train_dataset = train_datagen.flow_from_directory(
    data_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"  # Since we are dealing with two categories deathstar and notdeathstar we use binary classification
)


2025-03-11 09:43:40.108705: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/Image_identification/Train'

In [ ]:
# This is a pre-trained model that has already learned useful features from millions of images.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(150, 150, 3),  # The expected input size for MobileNetV2
    include_top=False,  # We don't need the original classification layers (we'll add our own)
    weights="imagenet"  # Use pre-trained weights from ImageNet (saves training time)
)
base_model.trainable = False  # Freeze the pre-trained layers so they don’t get updated


# Build the model - a custom CNN (Convolutional Neural Network) for image classification
model = keras.Sequential([
    keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(128, 128, 3)),  # First convolution layer with 64 filters
    keras.layers.MaxPooling2D(2,2),  # Reduces image size while keeping important features
    keras.layers.Conv2D(128, (3,3), activation='relu'),  # Second convolution layer with 128 filters
    keras.layers.MaxPooling2D(2,2),
    keras.layers.Conv2D(256, (3,3), activation='relu'),  # Third convolution layer with 256 filters
    keras.layers.MaxPooling2D(2,2),
    keras.layers.Flatten(),  # Converts the feature maps into a single 1D vector
    keras.layers.Dense(256, activation='relu'),  # Fully connected layer with 256 neurons
    keras.layers.Dropout(0.5),  # Drops 50% of neurons during training to prevent overfitting
    keras.layers.Dense(1, activation='sigmoid')  # Output layer (1 neuron, sigmoid activation for binary classification)
])

# Compile the model - setting up how it learns
model.compile(
    optimizer="adam",  # Adam optimizer (adapts learning rate automatically)
    loss="binary_crossentropy",  # Since we have two categories (binary classification)
    metrics=["accuracy"]  # Track accuracy during training
)

# Train the model using the prepared dataset
model.fit(train_dataset, epochs=20)  # Train for 20 iterations over the dataset (epochs)

# Save the trained model to Google Drive for later use
model.save("/content/drive/My Drive/Image_identification/Trained_classifier.h5")


**Loads a Pre-Trained Model** – It loads your saved model that identifies weak points in Death Star plans.

**Processes Input Images **– Resizes, converts to an array, normalizes, and expands dimensions to match the model's expected input.

**Makes Predictions** – Uses the model to classify the image as either "Death Star Weak Point" or "Not a Death Star."

**Saves Positive Detections** – If an image is classified as a weak point, it gets stored in a specific folder.

In [ ]:
import os
import numpy as np
import cv2  #uses openCV
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the trained deep learning model
# This model was previously trained to detect weak points in Death Star plans
model = load_model("/content/drive/My Drive/Image_identification/Trained_classifier.h5")

# Define the folder where detected images will be saved
detected_folder = "/content/drive/My Drive/Image_identification/detect_weak_points"

# Create the folder if it doesn’t already exist
os.makedirs(detected_folder, exist_ok=True)

def predict_and_save(image_path):
    """
    This function takes an image file, preprocesses it,
    feeds it into the trained model, and saves it if classified as 'Positive (Death Star)'.
    """

    IMG_SIZE = (128, 128)  # The model expects images of size 128x128 pixels

    # Load the image and resize it to the expected dimensions
    img = image.load_img(image_path, target_size=IMG_SIZE, color_mode="rgb")

    # Convert the image to an array format that the model can process
    img_array = image.img_to_array(img)

    # Expand dimensions to create a batch (since the model expects a batch input)
    img_array = np.expand_dims(img_array, axis=0)

    # Normalize pixel values to be between 0 and 1 (this improves model performance)
    img_array /= 255.0

    # Get prediction from the model
    prediction = model.predict(img_array)

    # If the model predicts a value below 0.9, classify it as a "Death Star" weak point
    # Otherwise, classify it as "Not a Death Star"
    label = "Positive (Death Star)" if prediction[0][0] < 0.9 else "Negative (Not a Death Star)"

    # If the image is classified as a weak point, save it in the detected folder
    if label == "Positive (Death Star)":
        filename = os.path.basename(image_path)  # Extract the image filename
        save_path = os.path.join(detected_folder, filename)  # Define where to save it
        img.save(save_path)  # Save the image in the designated folder

    return label  # Return the classification result


This code calls predict_and_save function in a loop top predict all the images in mix folder



In [ ]:
import os

# Folder containing images to classify
input_folder = "/content/drive/My Drive/Image_identification/Positive_negative_mix"  # Change this to your actual folder name

# Loop through all images in the folder
for filename in os.listdir(input_folder):
    image_path = os.path.join(input_folder, filename)

    # Ensure it's an image file
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        label = predict_and_save(image_path)
        print(f"{filename}: {label}")


In [ ]:
# Author Owen Francis
import cv2
import numpy as np
import os

# Print the current working directory (for debugging purposes)
print("Currently working directory", os.getcwd())

# Define input and output folder paths
input_folder = r"/content/drive/My Drive/Image_identification/Deathstar_mix/DeathStar"
output_folder= r"/content/drive/My Drive/Image_identification/Cropped_folder"

"""
Function used to process a folder of images, detecting red circles, crop the image to the red circle,
and then makes the background transparent before saving the output
"""
def crop_red_circle_from_folder():
    # Ensure the output folder exist
    os.makedirs(output_folder, exist_ok=True)

    # Loop through all files in the input folder
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(input_folder, filename)

            # Create the output filename by appending "_cropped.png"
            output_filename = os.path.splitext(filename)[0] + "_cropped.png"
            output_path = os.path.join(output_folder, output_filename)

            # Load the image with all channels (including alpha if present)
            image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
            if image is None:
                print(f"Error: unable to load image {filename}.")
                continue
            # Ensure the image has an alpha channel (transparency support)
            if image.shape[-1] == 3: #If only BGR (3 channels), add an alpha channel
                b, g, r = cv2.split(image)
                alpha = np.ones(b.shape, dtype=b.dtype) * 255 # Fully opaque
                image = cv2.merge((b, g, r, alpha))

            # Convert image to HSV color space for better red color detection
            hsv = cv2.cvtColor(image[:, :, :3], cv2.COLOR_BGR2HSV)

            # Define two ranges for detecting red color (since red wraps around in HSV)
            lower_red1 = np.array([0, 120, 70])
            upper_red1 = np.array([10, 255, 255])
            lower_red2 = np.array([170, 120, 70])
            upper_red2 = np.array([180, 255, 255])

            # Create masks for red color
            mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
            mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
            mask= cv2.bitwise_or(mask1, mask2) # Combine both masks

            # Find contours in the red mask
            contours, _ =cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if not contours:
                print(f"Error: unable to find contours in {filename}.")
                continue

            # Find the largest contour (assumed to be the main red circle)
            max_contour = max(contours, key=cv2.contourArea)

            # Get the minimum enclosing circle for the detected contour
            (x, y), radius = cv2.minEnclosingCircle(max_contour)
            x, y, radius = int(x), int(y), int(radius) # Convert to integer values

            # Define the bounding box for cropping
            x1,y1 = max(0, x - radius), max(0, y - radius)
            x2,y2 = min(image.shape[1], x + radius), min(image.shape[0], y + radius)

            # Ensure a valid cropping image
            cropped_image = image[y1:y2, x1:x2].copy()

            if cropped_image.size == 0:
                print(f"Error: unable to crop {filename}.")
                continue

            # Create a circular mask the same size as the cropped image
            mask = np.zeros((cropped_image.shape[0], cropped_image.shape[1]), dtype=np.uint8)
            circle_center = (cropped_image.shape[1] // 2, cropped_image.shape[0] // 2)
            cv2.circle(mask, circle_center, radius, 255, -1) # Draw a filled white circle

            # Apply the circular mask to the alpha channel
            b, g, r, a = cv2.split(cropped_image)
            a[mask == 0] = 0 # Set all pixels outside the circle to fully transparent
            cropped_rgba = cv2.merge((b, g, r, a))
            
            # To resize both the cropped image 128x128
            cropped_rgba = cv2.resize(cropped_rgba, (128, 128), interpolation=cv2.INTER_LANCZOS4)

            # Save the final cropped image with transparency as a PNG
            cv2.imwrite(output_path, cropped_rgba)
            print(f"saved image to {output_folder}.")
# Run the function to process the images
crop_red_circle_from_folder()
